# Manual ingest smoke #docs

Use this notebook section after changing ingest to exercise the app boundary directly with the configured `LangMatchAgent` and infrastructure adapters, inspect the database state, and clean up the temporary rows. Pytest remains the durable contract; this is a complete manual check.

In [1]:
from pathlib import Path
import sys

repo = Path.cwd()
if repo.name == "sandbox":
    repo = repo.parent

src = repo / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

repo

PosixPath('/Users/jonasmeddeb/templates/alexandria')

In [ ]:
from uuid import UUID

from sqlalchemy import delete, or_, select

from application.app import get_app
from infrastructure.model import DocumentRow, NodeRow, OutboxRow


SMOKE_DOC_ID = UUID("00000000-0000-0000-0000-000000001001")
SMOKE_PARENT_ID = UUID("00000000-0000-0000-0000-000000001002")
SMOKE_LEAF_ID = UUID("00000000-0000-0000-0000-000000001003")
SMOKE_SOURCE = "notebook://manual-ingest-smoke"
SMOKE_KEY = f"split.check:{SMOKE_LEAF_ID}"
SMOKE_NODE_IDS = (SMOKE_LEAF_ID, SMOKE_PARENT_ID)

app = get_app()
session = app.docs.session


print(app.settings.model_dump_json(indent=2))

{
  "app": {
    "app_env": "development",
    "app_name": "alexandria",
    "app_version": "0.1.0",
    "debug": false,
    "api_host": "localhost",
    "api_port": 8002,
    "mcp_host": "localhost",
    "mcp_port": 9002,
    "worker_poll_interval": 3
  },
  "index": {
    "split_cap": 5
  },
  "queue": {
    "backend": "memory",
    "max_pending": 10000,
    "batch_size": 25,
    "max_attempts": 3,
    "visibility_timeout_seconds": 300
  },
  "agents": {
    "langchain": {
      "model": "gpt-5-nano",
      "api_key": "**********",
      "api_base_url": "https://api.openai.com/v1",
      "temperature": 0.0,
      "timeout_seconds": 30
    },
    "match": {
      "prompt": "match",
      "max_candidates": 50
    },
    "search": {
      "prompt": "search",
      "max_candidates": 50,
      "max_results": 20
    },
    "split": {
      "make_prompt": "split_make",
      "pick_prompt": "split_pick",
      "min_children": 2,
      "max_children": 5
    }
  },
  "logging": {
    "level": 

In [3]:
def clean_smoke() -> None:
    session.rollback()
    session.execute(
        delete(edges).where(
            or_(
                edges.c.document_id == SMOKE_DOC_ID,
                edges.c.node_id.in_(SMOKE_NODE_IDS),
            ),
        ),
    )
    session.execute(delete(OutboxRow).where(OutboxRow.key == SMOKE_KEY))

    doc = session.get(DocumentRow, SMOKE_DOC_ID)
    if doc is not None:
        session.delete(doc)

    for node_id in SMOKE_NODE_IDS:
        node = session.get(NodeRow, node_id)
        if node is not None:
            session.delete(node)

    if app.outbox is None:
        app.queue.pending()

    session.commit()


clean_smoke()

In [4]:
# Create test data
doc = DocumentRow(
    id=SMOKE_DOC_ID,
    source=SMOKE_SOURCE,
    title="Notebook smoke leaf validation",
    summary="A temporary document created to validate LangMatchAgent-backed ingest.",
)

parent = NodeRow(
    id=SMOKE_PARENT_ID,
    name="Notebook smoke parent",
    description="Temporary parent for the manual ingest smoke check.",
)

leaf = NodeRow(
    id=SMOKE_LEAF_ID,
    parent_id=SMOKE_PARENT_ID,
    name="Notebook smoke leaf validation",
    description="Temporary leaf for LangMatchAgent-backed ingest validation.",
)

# Persist test nodes
session.add_all([parent, leaf])
session.flush([parent, leaf])

# Verify environment state
leaf_ids = {row.id for row in app.tree.leaves()}
assert SMOKE_LEAF_ID in leaf_ids, f"Smoke leaf {SMOKE_LEAF_ID} not found in tree."

# Execute ingestion and validate mapping
node = await app.ingest(doc)
session.flush()

if node.id != SMOKE_LEAF_ID:
    session.rollback()
    raise AssertionError(
        f"LangMatchAgent error: Expected node {SMOKE_LEAF_ID}, but picked {node.id}"
    )

# Successful completion
print(
    f"Smoke test passed! LangMatchAgent correctly matched the leaf node. "
    f"Details: {{"
    f"'doc_id': '{doc.id}', "
    f"'match_agent': '{type(app.match_agent).__name__}', "
    f"'selected_node_id': '{node.id}'"
    f"}}"
)

[12:10:57] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Smoke test passed! LangMatchAgent correctly matched the leaf node. Details: {'doc_id': '00000000-0000-0000-0000-000000001001', 'match_agent': 'LangMatchAgent', 'selected_node_id': '00000000-0000-0000-0000-000000001003'}


In [5]:
# Verify persistence and associations
stored_doc = app.docs.get(SMOKE_DOC_ID)
linked_docs = app.tree.docs(leaf)

# Check database edges
edge_query = select(edges).where(
    edges.c.document_id == SMOKE_DOC_ID, 
    edges.c.node_id == SMOKE_LEAF_ID
)
edge_results = session.execute(edge_query).all()

# Determine queued node IDs
outbox_query = select(OutboxRow).where(OutboxRow.key == SMOKE_KEY)
outbox_rows = session.scalars(outbox_query).all()
queued_ids = [UUID(str(row.payload["node_id"])) for row in outbox_rows]

# Assertions
assert node.id == SMOKE_LEAF_ID, f"Expected node {SMOKE_LEAF_ID}, got {node.id}"
assert stored_doc is not None, "Document failed to persist."
assert [d.id for d in linked_docs] == [SMOKE_DOC_ID], "Linked documents mismatch."
assert len(edge_results) == 1, f"Expected 1 edge, found {len(edge_results)}."
assert queued_ids == [SMOKE_LEAF_ID], f"Queue mismatch. Expected {[SMOKE_LEAF_ID]}, got {queued_ids}"

session.commit()

# Return status
status = {
    "stored_doc": stored_doc.title,
    "linked_doc_ids": [str(d.id) for d in linked_docs],
    "queued_node_ids": [str(n) for n in queued_ids],
    "committed": True
}
print(f"Verification complete. Results: {status}")

AssertionError: Queue mismatch. Expected [UUID('00000000-0000-0000-0000-000000001003')], got []

In [6]:
clean_smoke()
assert app.docs.get(SMOKE_DOC_ID) is None
assert session.get(NodeRow, SMOKE_PARENT_ID) is None
assert session.get(NodeRow, SMOKE_LEAF_ID) is None
assert session.scalar(select(OutboxRow).where(OutboxRow.key == SMOKE_KEY)) is None

"manual ingest smoke data removed"

'manual ingest smoke data removed'